In [6]:
import pandas as pd
df = pd.read_csv(
    "/Users/thisisjasmine/Desktop/ECO225/sfpd_with_sun_and_vod4.csv",
    low_memory=False
)
df["perceived_race_ethnicity_code"] = pd.to_numeric(df["perceived_race_ethnicity_code"], errors="coerce")

# 3. Remove Hispanic (code = 3)
df_cleaned = df[df["perceived_race_ethnicity_code"] != 3]  # Remove all Hispanic entries

# 4. Remove any multi-code entries (if any exist)
df_cleaned = df_cleaned[~df_cleaned["perceived_race_ethnicity_code"].astype(str).str.contains(",")]
df_cleaned = df_cleaned.dropna(subset=["perceived_race_ethnicity_code", "stop_datetime"])
# 5. Verify the cleaned data
print(df_cleaned["perceived_race_ethnicity_code"].value_counts())

# 6. Save the cleaned data
df_cleaned.to_csv("cleaned_sfpd_data.csv", index=False)


perceived_race_ethnicity_code
7.0    98285
2.0    70311
1.0    32877
4.0    18852
6.0     3529
5.0      416
Name: count, dtype: int64


In [8]:
import pandas as pd
import numpy as np

# 1) Load
pop = pd.read_csv("/Users/thisisjasmine/Desktop/Race and Ethnicity.csv")

# 2. Clean basic data
pop["Ethnicity"] = pop["Ethnicity"].astype(str).str.strip()
pop["Race"] = pop["Race"].astype(str).str.strip()
pop["Year"] = pd.to_numeric(pop["Year"], errors="coerce")
pop["Population"] = pd.to_numeric(pop["Population"], errors="coerce")

pop = pop.dropna(subset=["Year", "Race", "Ethnicity", "Population"]).copy()

# 3. Remove "Hispanic" ethnicity and keep the race info
# We are removing all entries with "Hispanic or Latino" in the Ethnicity column
pop = pop[pop["Ethnicity"] != "Hispanic or Latino"].copy()

# 4. Remove "Two Races" and "Other" categories
exclude_races = {
    "Other",
    "Two Races Including Other",
    "Two Races Excluding Other & Three or More Races"
}
pop = pop[~pop["Race"].isin(exclude_races)].copy()

# 5. Collapse the data to Year × Race (we are now focusing on the Race category)
pop_clean = (
    pop.groupby(["Year", "Race"], as_index=False)
       .agg(Population=("Population", "sum"))
)

# 6. Create within-year share (proportion of each race group in that year)
pop_clean["pop_share"] = (
    pop_clean["Population"] /
    pop_clean.groupby("Year")["Population"].transform("sum")
)

# 7. Map to your main dataset race codes (1–7)
race_to_code = {
    "Asian": 1,
    "Black or African American": 2,
    "American Indian & Alaska Native": 5,
    "Native Hawaiian & Other Pacific Islander": 6,
    "White": 7
}

pop_clean["perceived_race_ethnicity_code"] = (
    pop_clean["Race"].map(race_to_code)
)

# 8. Drop rows with NA codes (if there are any residual unmapped races)
pop_clean = pop_clean.dropna(subset=["perceived_race_ethnicity_code"]).copy()

# 9. Convert to integer type for the code
pop_clean["perceived_race_ethnicity_code"] = pop_clean["perceived_race_ethnicity_code"].astype(int)

# 10. Final checks
print(pop_clean.head(12))

# 11. Save the cleaned dataset
pop_clean.to_csv("pop_clean_no_hispanic.csv", index=False)


    Year                                      Race  Population  pop_share  \
0   2018           American Indian & Alaska Native        1363   0.001863   
1   2018                                     Asian      294846   0.402949   
2   2018                 Black or African American       43619   0.059612   
3   2018  Native Hawaiian & Other Pacific Islander        2694   0.003682   
5   2018                                     White      353670   0.483341   
6   2019           American Indian & Alaska Native        1634   0.002219   
7   2019                                     Asian      298108   0.404863   
8   2019                 Black or African American       43782   0.059461   
9   2019  Native Hawaiian & Other Pacific Islander        2934   0.003985   
11  2019                                     White      354423   0.481345   
12  2020           American Indian & Alaska Native        1796   0.002447   
13  2020                                     Asian      297279   0.405038   

In [7]:
import pandas as pd

# 1. Load the stop data
df = pd.read_csv("/Users/thisisjasmine/Desktop/ECO225/cleaned_sfpd_data.csv")

# 2. Ensure that perceived_race_ethnicity_code is numeric
df["perceived_race_ethnicity_code"] = pd.to_numeric(df["perceived_race_ethnicity_code"], errors="coerce")

# 3. Remove Hispanic (code = 3) and multi-code entries (if any exist)
df_cleaned = df[df["perceived_race_ethnicity_code"] != 3]  # Remove all Hispanic entries
df_cleaned = df_cleaned[~df_cleaned["perceived_race_ethnicity_code"].astype(str).str.contains(",")]  # Remove multi-codes
df_cleaned = df_cleaned.dropna(subset=["perceived_race_ethnicity_code", "stop_datetime"])  # Remove blank rows

# 4. Extract year from stop_datetime column
df_cleaned["stop_datetime"] = pd.to_datetime(df_cleaned["stop_datetime"], errors="coerce")
df_cleaned["Year"] = df_cleaned["stop_datetime"].dt.year

# 5. Filter the data for years 2018–2025
df_filtered = df_cleaned[(df_cleaned["Year"] >= 2018) & (df_cleaned["Year"] <= 2025)].copy()

# 6. Group by Year and perceived_race_ethnicity_code to get counts
yearly_counts = df_filtered.groupby(["Year", "perceived_race_ethnicity_code"]).size().reset_index(name="count")

# 7. Calculate the proportion of each race code within each year
yearly_counts["stop_share"] = (
    yearly_counts["count"] /
    yearly_counts.groupby("Year")["count"].transform("sum")
)

# 8. Save the result to CSV (optional)
yearly_counts.to_csv("yearly_ethnic_code_proportion.csv", index=False)

# 9. Verify the cleaned data
print(yearly_counts.head(20))


    Year  perceived_race_ethnicity_code  count  stop_share
0   2018                            1.0   5548    0.135133
1   2018                            2.0  13318    0.324386
2   2018                            4.0   3304    0.080475
3   2018                            5.0     81    0.001973
4   2018                            6.0    643    0.015662
5   2018                            7.0  18162    0.442371
6   2019                            1.0  10900    0.147351
7   2019                            2.0  22532    0.304598
8   2019                            4.0   6867    0.092831
9   2019                            5.0    128    0.001730
10  2019                            6.0   1039    0.014046
11  2019                            7.0  32507    0.439444
12  2020                            1.0   3521    0.128167
13  2020                            2.0   9563    0.348100
14  2020                            4.0   1815    0.066067
15  2020                            5.0     68    0.0024